In [1]:
import random
import networkx as nx

spl = nx.shortest_path_length
def SPL(g):
    if not nx.is_connected(g): raise Exception ('g is not connected!')
        # '''g is disconnected! We consider its largest connected component instead!'''
        # largest_cc = max(nx.connected_components(g), key=len)
        # g = g.subgraph(largest_cc)
    spl_dic = dict() 
    V = list(g.nodes())
    V.sort()
    for p in V:
        for q in V:
            if (q,p) in spl_dic:
                d = spl_dic[(q,p)]
            else:
                d = nx.shortest_path_length(g,p,q)
            spl_dic[(p,q)] = d
    return spl_dic
            
def is_complete(tau, tokenset):
    if len(tau) == len(tokenset):
        return True
    return False

# Match a node for a not yet mapped token
def map_completion_all(tau, dag, AG):
    for node in dag.topological_op_nodes():
        if len(tau) == len(dag.qubits):
            return tau
        if not isinstance(node, DAGOpNode):
            continue
        token1, token2 = node.qargs[0]._index, node.qargs[1]._index
        if token1 in tau:
            if token2 in tau:
                continue
            # map token1 to a neighbour of tau[token2]
            u = tau[token1]
            Candidates = [[v, SPL(AG)[(u,v)]] for v in AG.nodes() if v not in tau.values()]
            tau[token2] = sorted(Candidates, key=lambda x: x[1])[0][0]
        elif token2 in tau:
            # map token1 to a neighbour of tau[token2]
            u = tau[token2]
            Candidates = [[v, SPL(AG)[(u,v)]] for v in AG.nodes() if v not in tau.values()]
            tau[token1] = sorted(Candidates, key=lambda x: x[1])[0][0]
        else:
            # map them to an edge if possible
            edge_dist = 2*nx.diameter(AG)
            for edge in AG.edges:
                if edge[0] in tau.values() or edge[1] in tau.values():
                    continue
                u, v = edge
                # Compute shortest path lengths from u and v to occupied nodes
                shortest_paths_u = nx.single_source_shortest_path_length(AG, u)
                shortest_paths_v = nx.single_source_shortest_path_length(AG, v)

                # Filter shortest paths for occupied nodes
                shortest_paths_u_filtered = {node: distance for node, distance in shortest_paths_u.items() if node in tau.values()}
                shortest_paths_v_filtered = {node: distance for node, distance in shortest_paths_v.items() if node in tau.values()}

                # Find the minimum shortest path length for each node
                min_u = min(shortest_paths_u_filtered.values())
                min_v = min(shortest_paths_v_filtered.values())

                # Find the overall minimum shortest path length
                min_length = min_u + min_v
                if min_length < edge_dist:
                    edge_dist = min_length
                    tau[token1], tau[token2] = u, v
                
def one_shot_map_extension(tau, tokenset, AG):
    '''Extend current mapping (dic1) one step '''
    
    if is_complete(tau, tokenset):
        print('The mapping is already complete!')
        return tau

    UnOcc = list(v for v in AG.nodes() if v not in tau.values())
    Occ = list(v for v in AG.nodes() if v in tau.values())
    Occ2 = list(u for u in UnOcc if min([SPL(AG)[(u,v)] for v in Occ]) <= 2)
    print(UnOcc, Occ, Occ2)
    
    q = random.choice([p for p in tokenset if p not in tau])  
    u = random.choice(Occ2)  
    tau[q] = u
    return tau

def map_completion(tau, tokenset, AG):
    """tau is a token-to-node mapping or a logical-to-physical mapping"""
    
    if is_complete(tau, tokenset):
        return tau
    while True:
        tau = one_shot_map_extension(tau, tokenset, AG)
        if is_complete(tau, AG):
            return tau

def random_map_completion(tau, tokenset, AG):
    """tau is a token-to-node mapping or a logical-to-physical mapping"""
    
    while True:
        if is_complete(tau, tokenset):
            return tau
        UnMappedToken = [x for x in tokenset if x not in tau]
        UnOccupiedNode = [n for n in AG.nodes() if n not in tau.values()]
        #print(UnMappedToken, UnOccupiedNode, tau)
        x = random.choice(UnMappedToken)
        n = random.choice(UnOccupiedNode)
        newtau = copy.copy(tau)
        newtau[x] = n
        tau = copy.copy(newtau)
        
# Check if tau can execute every cx gate in dag
def verify(tau, dag, AG):
    for gate in dag.nodes():
        if type(gate) != DAGOpNode:
            continue
        p, q = gate.qargs[0]._index, gate.qargs[1]._index
        if (tau[p], tau[q]) not in AG.edges():
            return False
    return True


# Partition and Route

Use partition function we partition an input circuit into sections $C_1,\ldots,C_m$. For $C_1$, we output an arbitrary embedding $\tau_1$, and transform all $C_i$ as $\tau_1(C_i)$. Thus $\tau_1(C_1)$ has initial mapping $id$ and the interaction grpah of $\tau_1(C_2)$ is $IG(\tau_1(C_2))=\tau_1(IG(C_2))$, which we regard as the constraints, call the connect_two function, and obtain the new mapping $\sigma_2$, which embeds $\tau_1(IG(C_2))$ to $AG$. 

In [2]:
import random
from ag import *
from connect_two import *
from dac_part import *

"""Test DAC"""
path = '../bench/qiskit_circuit_benchmark/' 
#path = '../bench/20Q_gate_Tokyo/' 
#filename = '20QBT_gate_Tokyo_large_opt1_0_1.5_no.8.qasm'
#filename = 'excitation_preserving_6.qasm'
#filename = 'grover_operator_10.qasm'
filename = 'qft_10.qasm'

print(f'filename: {filename}')
#AG = qgrid(2,3)
AG = q20()
#AG = qgrid(6,9)

with open(path+filename, 'r') as file:
    qasm_code = file.read()

# Create a QuantumCircuit from the QASM code
qc = QuantumCircuit.from_qasm_str(qasm_code)

#print(qc.count_ops(), dir(qc))
#print(list(qc._qubits))

newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)

tokenset = set([q._index for q in newcirc.qubits])
print(f'Tokenset {tokenset} is the set of qubits in the reduced input circuit.')

dag = circuit_to_dag(newcirc)
print(dag.count_ops())

X = partition(dag, AG, method='greedy')
print('xx')
IniDag = copy.deepcopy(X[0])
#W = copy.deepcopy(X)
#for i in range(len(W)-1):
#    W[i], W[i+1] = realign(W[i], W[i+1])
#X = copy.deepcopy(W)

#print(f'The reduced circuit is partitioned in the following sections:')
#for i in range(len(X)):
#    print([(gate.qargs[0]._index, gate.qargs[1]._index) for gate in X[i].nodes() if type(gate)==DAGOpNode])

# create an initial mapping, which is a token-to-node or logical-to-physical mapping
cur_graph = graph_of_circuit(dag_to_circuit(IniDag))
#cur_graph = graph_of_circuit(dag_to_circuit(X[0]))
A, inimap = is_embeddable(cur_graph, AG, 10)
print(f'*The selected initial mapping is {inimap}')
if not A:
    raise Exception('The first graph should be embeddable!')

if is_complete(inimap, tokenset):
    tau = copy.copy(inimap)
else:
    #TODO: Devise a better completion method
    #tau =  random_map_completion(inimap, tokenset, AG)
    tau = map_completion_all(inimap, dag, AG)
    
print(f'**The completed initial mapping is: {tau}')
cur_map = copy.copy(tau)
print(f'The current map satisfies? [{verify(cur_map, X[0], AG)}] all gates in the current section')

filename: qft_10.qasm
Tokenset {0, 1, 2, 3, 4, 5, 6, 7, 8, 9} is the set of qubits in the reduced input circuit.
{'cx': 50}
The dagCircuit has {'cx': 50} cx gates
step 7: Section 1 constructed!
step 13: Section 2 constructed!
step 18: Section 3 constructed!
step 22: Section 4 constructed!
The partition is complete and we have 5 sections!
xx
*The selected initial mapping is {9: 13, 6: 12, 7: 8, 8: 7, 3: 18, 4: 14, 5: 19}
**The completed initial mapping is: {9: 13, 6: 12, 7: 8, 8: 7, 3: 18, 4: 14, 5: 19, 2: 1, 1: 2, 0: 3}
The current map satisfies? [True] all gates in the current section


In [ ]:
"""This part is extremely slow for excitation_preserving_8.qasm as its actions often require 5 or more SWAPs"""

import time
print(f'filename: {filename}')
map_id ={v:v for v in AG.nodes()}
# route as you go
ACTION = dict()
for i in range(1,len(X)):
    cur_graph = graph_of_circuit(dag_to_circuit(X[i]))
    EX = cur_graph.edges()
    print(f'step {i}: the current mapping is {cur_map}')
    #print(f'step {i}: the curren constraints are {EX}')
    constraints = [(cur_map[edge[0]],cur_map[edge[1]]) for edge in EX]
    print(f'step {i}: the mapped constraints are {constraints}')
    
    #TODO iterative_dfs is slow
    action = iterative_dfs(map_id,constraints,AG)
    
    ACTION[i] = action
    print(f'step {i}: action {action}')
    
    if action == None:
        print(f'!! Check why the action is {action}!')
        continue
    for edge in action:
        cur_map = swap(edge, cur_map, AG)
    print(f'The current map satisfies? [{verify(cur_map, X[i], AG)}] all gates in the current section')
    
cost = sum([len(action) for action in ACTION.values()])

print(f'{cost} swaps are added! \n {ACTION}')

filename: qft_10.qasm
step 1: the current mapping is {9: 13, 6: 12, 7: 8, 8: 7, 3: 18, 4: 14, 5: 19, 2: 1, 1: 2, 0: 3}
step 1: the mapped constraints are [(7, 19), (7, 14), (7, 18), (7, 1), (7, 2), (19, 8), (19, 12), (8, 14), (13, 1), (13, 2), (13, 3)]
test (0, False)
test (1, False)
test (2, False)
test (3, False)
test (4, False)
test (5, False)
